In [1]:
import sys, importlib, os
repo_start = f'../'
sys.path.append(repo_start)

from modules.utils.imports import *
from modules.utils.numpy_torch_conversion import *
from modules.binn.build_binns import BINN
from modules.loaders.format_data import format_data_general
from modules.generate_data.simulate_system import *
from modules.binn.model_wrapper import model_wrapper
from modules.analysis.visualize_surface import visualize_surface
from modules.analysis.simulate_surface import simulate_surface
import modules.generate_data.simulate_system


In [2]:
dir_name = '/work/users/s/m/smyersn/elston/projects/kinetics_binns/testing/turing_type_2/0.1_0.5_5_5/no_diffusion'

In [3]:
def to_torch(ndarray):
    arr = torch.tensor(ndarray, dtype=torch.float)
    arr.requires_grad_(True)
    arr = arr.to(device)
    return arr

# load params from configuration file
config = {}
exec(Path(f'{dir_name}/config.cfg').read_text(encoding="utf8"), {}, config)

training_data_path = config['training_data_path']

reaction = getattr(modules.generate_data.simulate_system,
                   config['reaction'])
params = [float(param) for param in config['params'].replace(',', ' ').split()]

dimensions = int(config['dimensions'])
species = int(config['species'])

density_weight = int(config['density_weight'])

uv_layers = int(config['uv_layers'])
uv_neurons = int(config['uv_neurons'])
f_layers = int(config['f_layers'])
f_neurons = int(config['f_neurons'])

epsilon = float(config['epsilon'])
points = int(config['points'])

diffusion = bool(config['diffusion'])

uv_arch = uv_layers * [uv_neurons] + [2]
f_arch = f_layers * [f_neurons] + [1]


In [4]:
# initialize model
device = torch.device('cpu')

binn = BINN(
    species=species, 
    dimensions=dimensions,
    uv_arch=uv_arch, 
    f_arch=f_arch,
    diff_coeffs=params[:2],
    diff=diffusion)

binn.to(device)

opt = torch.optim.Adam(list(binn.parameters()), lr=1e-3)
model = model_wrapper(
    model=binn,
    optimizer=opt,
    loss=binn.loss,
    augmentation=None,
    save_name=f'{dir_name}/binn')

model.load(f"{dir_name}/binn_best_val_model", device=device)

In [5]:
def visualize_surface(model, dimensions, species, reaction, params, model_dir, training_data_path):
    
    # Load and format training data
    training_data = format_data_general(dimensions, species, training_data_path)
    u = training_data[:, dimensions+1]
    v = training_data[:, dimensions+2]
    u_triangle_mesh, v_triangle_mesh = lltriangle(u, v)

    # u_points = np.linspace(np.min(u), np.max(u), 501)
    # v_points = np.linspace(np.min(v), np.max(v), 501)

    # u_triangle_mesh, v_triangle_mesh = np.meshgrid(u_points, v_points)

    # Calculate true reaction surface
    F_true = reaction(u_triangle_mesh, v_triangle_mesh, params[2:])

    # Calculate MLP reaction surface
    uv = np.column_stack((np.ravel(u_triangle_mesh), np.ravel(v_triangle_mesh)))
    F_mlp_stacked = to_numpy(model.model.reaction(to_torch(uv)[:, None]))
    F_mlp = np.reshape(F_mlp_stacked, (501, 501))

    # Plot
    fig = make_subplots(rows=1, cols=2,
                        specs=[[{'type':'scene'}, {'type':'scene'}]],
                        subplot_titles=('F(u, v)', 'F*(u, v)'),
                        horizontal_spacing = 0)

    fig.layout.annotations[0].update(y=0.8)
    fig.layout.annotations[1].update(y=0.8)
    fig.update_annotations(font_size=24, font_color='#000000')

    fig.add_trace(
        go.Surface(z=F_true, x=u_triangle_mesh, y=v_triangle_mesh,
                                    colorscale='mint',
                                    showscale=False,
                                    colorbar=dict(
                                        x=1.15,
                                        title='True',
                                        len=0.5)),
        row=1, col=1)
    fig.add_trace(
        go.Surface(z=F_mlp, x=u_triangle_mesh, y=v_triangle_mesh,
                                    colorscale='mint',
                                    showscale=False,
                                    colorbar=dict(
                                        title='MLP',
                                        len=0.5)),
        row=1, col=2)

    fig.update_layout(autosize=True,
        width=1500, 
        height=800,
        font=dict(color = '#000000',
                size=20),
        
        scene=dict( 
        xaxis_title='[u] (uM)',
        yaxis_title='[v] (uM)',
        zaxis_title='F',
        xaxis = dict(
            tick0 = 0,
            dtick = 2,
            tickfont = dict(size=18)),
        yaxis = dict(
            tick0 = 0.2,
            dtick = 0.4,
            tickfont = dict(size=18)),
        zaxis = dict(
            tick0 = 0,
            dtick = 2,
            tickfont = dict(size=18),
            range=[-4, 11]),

        camera=dict(eye=dict(x=1, y=-2.5, z=1)),),
                    
        scene2=dict(
        xaxis_title='[u] (uM)',
        yaxis_title='[v] (uM)',
        zaxis_title='F*',
        xaxis = dict(
            tick0 = 0,
            dtick = 2,
            tickfont = dict(size=18)),
        yaxis = dict(
            tick0 = 0.2,
            dtick = 0.4,
            tickfont = dict(size=18)),
        zaxis = dict(
            tick0 = 0,
            dtick = 2,
            tickfont = dict(size=18),
            range=[-4, 11]),

        camera=dict(eye=dict(x=1, y=-2.5, z=1))))

    fig.update_coloraxes(showscale=False)

    fig.show()
    # fig.write_image(f'{model_dir}/f_mlp_surface.png')

In [ ]:
visualize_surface(model, dimensions, species, reaction, params, dir_name, training_data_path)